# I. Import Library


In [1]:
!pip install py7zr
import pandas as pd
import numpy as np
import os
import shutil
import py7zr
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np



# II. Load Dataset
## 1.Convert to CSV


In [2]:

input_dir = "/kaggle/input/competitions/favorita-grocery-sales-forecasting/"
output_dir = "/kaggle/working/extractedCSV"

# 1. Clean previous extraction (prevents duplication issues)
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.makedirs(output_dir, exist_ok=True)

# 2. Extract all .7z files
for file in os.listdir(input_dir):
    if file.endswith(".7z"):
        file_path = os.path.join(input_dir, file)

        with py7zr.SevenZipFile(file_path, 'r') as archive:
            archive.extractall(path=output_dir)

        print("Extracted:", file)

Extracted: test.csv.7z
Extracted: stores.csv.7z
Extracted: items.csv.7z
Extracted: holidays_events.csv.7z
Extracted: transactions.csv.7z
Extracted: train.csv.7z
Extracted: oil.csv.7z
Extracted: sample_submission.csv.7z


## 2.Load CSV files into the Dataframe

In [3]:

train = pd.read_csv(output_dir+"/train.csv")
stores = pd.read_csv(output_dir +"/stores.csv")
items = pd.read_csv(output_dir +"/items.csv")
holidays = pd.read_csv(output_dir +"/holidays_events.csv")
# basic inspection of the data
print(train.shape)
print(train.head())

train.info()
train.isnull().sum()

/tmp/ipykernel_288/214754714.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(output_dir+"/train.csv")


(125497040, 6)
   id        date  store_nbr  item_nbr  unit_sales onpromotion
0   0  2013-01-01         25    103665         7.0         NaN
1   1  2013-01-01         25    105574         1.0         NaN
2   2  2013-01-01         25    105575         2.0         NaN
3   3  2013-01-01         25    108079         1.0         NaN
4   4  2013-01-01         25    108701         1.0         NaN
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125497040 entries, 0 to 125497039
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   item_nbr     int64  
 4   unit_sales   float64
 5   onpromotion  object 
dtypes: float64(1), int64(3), object(2)
memory usage: 5.6+ GB


id                    0
date                  0
store_nbr             0
item_nbr              0
unit_sales            0
onpromotion    21657651
dtype: int64

## 3. Clean the train Dataset

In [4]:


# convert date columns to datetime
train['date'] = pd.to_datetime(train['date'])
# get data last 1 year
cutoff = train['date'].max() - pd.Timedelta(days=365)
train = train[train['date'] >= cutoff]
# handle negative sales 
train['unit_sales'] = train['unit_sales'].apply(lambda x: max(x, 0))
# log-transform sales to reduce skewness
train['unit_sales'] = np.log1p(train['unit_sales'])


## 4. Clean stores Dataset

In [5]:
# explore table
print(stores.info())
stores.isnull().sum()
display(stores.head())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   store_nbr  54 non-null     int64 
 1   city       54 non-null     object
 2   state      54 non-null     object
 3   type       54 non-null     object
 4   cluster    54 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 2.2+ KB
None


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


## 5. Clean items Dataset



In [6]:
# explore table
print(items.info())
display(items.head())
# check for missing values in the other datasets
print(items.isnull().sum())
# convert categorical columns to category dtype for memory efficiency
items['family'] = items['family'].astype('category')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4100 entries, 0 to 4099
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   item_nbr    4100 non-null   int64 
 1   family      4100 non-null   object
 2   class       4100 non-null   int64 
 3   perishable  4100 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 128.3+ KB
None


,item_nbr,family,class,perishable
0,96995,GROCERY I,1093,0
1,99197,GROCERY I,1067,0
2,103501,CLEANING,3008,0
3,103520,GROCERY I,1028,0
4,103665,BREAD/BAKERY,2712,1


item_nbr      0
family        0
class         0
perishable    0
dtype: int64


## 6. Clean holidays_events Dataset


In [7]:
# explore table'
print(holidays.info())
display(holidays.head())
print(holidays.isnull().sum())
print(f"Duplicated days: {holidays['date'].duplicated().sum()}")
# convert date columns to datetime
holidays['date'] = pd.to_datetime(holidays['date'])
# keep only relevant holidays
holidays = holidays[holidays['transferred'] == False]
# simplify holiday
holidays['is_holiday'] = 1
holidays = holidays[['date', 'is_holiday']]
display(holidays.head())
# drop duplicate holidays
holidays = holidays.drop_duplicates('date')



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         350 non-null    object
 1   type         350 non-null    object
 2   locale       350 non-null    object
 3   locale_name  350 non-null    object
 4   description  350 non-null    object
 5   transferred  350 non-null    bool  
dtypes: bool(1), object(5)
memory usage: 14.1+ KB
None


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64
Duplicated days: 38


,date,is_holiday
0,2012-03-02,1
1,2012-04-01,1
2,2012-04-12,1
3,2012-04-14,1
4,2012-04-21,1


## 7. Merge All Data


In [8]:
# Merge stores
data = train.merge(stores, on='store_nbr', how='left')
print(data.duplicated(['date','store_nbr','item_nbr']).sum())
# Merge items
data = data.merge(items, on='item_nbr', how='left')
print(data.duplicated(['date','store_nbr','item_nbr']).sum())
# Merge holidays
data = data.merge(holidays, on='date', how='left')
print(data.duplicated(['date','store_nbr','item_nbr']).sum())
# explore merged data
print("Data info:")
data.info()
print("First few rows of the merged data:")
display(data.head())
print("Missing values in the merged data:")
print(data.isnull().sum())
# fill missing values

data['is_holiday'] = data['is_holiday'].fillna(0)
data['onpromotion'] = data['onpromotion'].fillna("None")

0
0
0
Data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37454835 entries, 0 to 37454834
Data columns (total 14 columns):
 #   Column       Dtype         
---  ------       -----         
 0   id           int64         
 1   date         datetime64[ns]
 2   store_nbr    int64         
 3   item_nbr     int64         
 4   unit_sales   float64       
 5   onpromotion  object        
 6   city         object        
 7   state        object        
 8   type         object        
 9   cluster      int64         
 10  family       category      
 11  class        int64         
 12  perishable   int64         
 13  is_holiday   float64       
dtypes: category(1), datetime64[ns](1), float64(2), int64(6), object(4)
memory usage: 3.7+ GB
First few rows of the merged data:


,id,date,store_nbr,item_nbr,unit_sales,onpromotion,city,state,type,cluster,family,class,perishable,is_holiday
0,88042205,2016-08-15,1,103665,0.693147,False,Quito,Pichincha,D,13,BREAD/BAKERY,2712,1,1.0
1,88042206,2016-08-15,1,105574,0.693147,False,Quito,Pichincha,D,13,GROCERY I,1045,0,1.0
2,88042207,2016-08-15,1,105575,2.995732,False,Quito,Pichincha,D,13,GROCERY I,1045,0,1.0
3,88042208,2016-08-15,1,105577,0.693147,False,Quito,Pichincha,D,13,GROCERY I,1045,0,1.0
4,88042209,2016-08-15,1,105693,0.693147,False,Quito,Pichincha,D,13,GROCERY I,1034,0,1.0


Missing values in the merged data:
id                    0
date                  0
store_nbr             0
item_nbr              0
unit_sales            0
onpromotion           0
city                  0
state                 0
type                  0
cluster               0
family                0
class                 0
perishable            0
is_holiday     32655087
dtype: int64


/tmp/ipykernel_288/221962269.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['onpromotion'] = data['onpromotion'].fillna("None")


## 8 Features Engineering

In [9]:
# 1. Sort data by date for time series feature engineering
data = data.sort_values(['store_nbr', 'item_nbr', 'date'])

# 2. Extract date features
data['year'] = data['date'].dt.year
data['month'] = data['date'].dt.month
data['day'] = data['date'].dt.day
data['dayofweek'] = data['date'].dt.dayofweek

# 3. Lag features
data['lag_7'] = data.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(7)
data['lag_14'] = data.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(14)
#print(data.duplicated(['date','store_nbr','item_nbr']).sum())

# 4. Rolling mean (no leakage)
data['rolling_mean_7'] = (data.groupby(['store_nbr', 'item_nbr'])['unit_sales'].transform(lambda x: x.shift(1).rolling(7).mean()))
#print(data.duplicated(['date','store_nbr','item_nbr']).sum())
#print(data.shape)

# 5. DROP missing values 
data = data.dropna(subset=['lag_7', 'lag_14', 'rolling_mean_7'])
# 6. Encode categorical variables (AFTER feature engineering)
data = pd.get_dummies(data,columns=['family', 'city', 'state', 'type'],drop_first=True
)
#print(data.shape)
#print(data.duplicated(['date','store_nbr','item_nbr']).sum())

In [10]:
"""
# 7. Save final dataset
os.makedirs("/kaggle/working/cleanedCSV", exist_ok=True)
data_cleaned_path = "/kaggle/working/cleanedCSV/"
data.to_csv(data_cleaned_path +"data_cleaned.csv", index=False)
"""

'\n# 7. Save final dataset\nos.makedirs("/kaggle/working/cleanedCSV", exist_ok=True)\ndata_cleaned_path = "/kaggle/working/cleanedCSV/"\ndata.to_csv(data_cleaned_path +"data_cleaned.csv", index=False)\n'

In [11]:
"""
# Zip data to download
!zip -r hazards_results.zip /kaggle/working/cleanedCSV
from IPython.display import FileLink
FileLink(r'hazards_results.zip')
"""

"\n# Zip data to download\n!zip -r hazards_results.zip /kaggle/working/cleanedCSV\nfrom IPython.display import FileLink\nFileLink(r'hazards_results.zip')\n"

In [12]:
# view head of the data
pd.set_option('display.max_columns', None)
display(data.head())
print(data.shape)

,id,date,store_nbr,item_nbr,unit_sales,onpromotion,cluster,class,perishable,is_holiday,year,month,day,dayofweek,lag_7,lag_14,rolling_mean_7,family_BABY CARE,family_BEAUTY,family_BEVERAGES,family_BOOKS,family_BREAD/BAKERY,family_CELEBRATION,family_CLEANING,family_DAIRY,family_DELI,family_EGGS,family_FROZEN FOODS,family_GROCERY I,family_GROCERY II,family_HARDWARE,family_HOME AND KITCHEN I,family_HOME AND KITCHEN II,family_HOME APPLIANCES,family_HOME CARE,family_LADIESWEAR,family_LAWN AND GARDEN,family_LINGERIE,"family_LIQUOR,WINE,BEER",family_MAGAZINES,family_MEATS,family_PERSONAL CARE,family_PET SUPPLIES,family_PLAYERS AND ELECTRONICS,family_POULTRY,family_PREPARED FOODS,family_PRODUCE,family_SCHOOL AND OFFICE SUPPLIES,family_SEAFOOD,city_Babahoyo,city_Cayambe,city_Cuenca,city_Daule,city_El Carmen,city_Esmeraldas,city_Guaranda,city_Guayaquil,city_Ibarra,city_Latacunga,city_Libertad,city_Loja,city_Machala,city_Manta,city_Playas,city_Puyo,city_Quevedo,city_Quito,city_Riobamba,city_Salinas,city_Santo Domingo,state_Bolivar,state_Chimborazo,state_Cotopaxi,state_El Oro,state_Esmeraldas,state_Guayas,state_Imbabura,state_Loja,state_Los Rios,state_Manabi,state_Pastaza,state_Pichincha,state_Santa Elena,state_Santo Domingo de los Tsachilas,state_Tungurahua,type_B,type_C,type_D,type_E
30825298,118867503,2017-06-14,1,96995,0.693147,False,13,1093,0,0.0,2017,6,14,2,0.693147,1.098612,0.693147,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
31978040,120020245,2017-06-25,1,96995,0.693147,False,13,1093,0,1.0,2017,6,25,6,0.693147,0.693147,0.693147,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
32395968,120438173,2017-06-29,1,96995,0.693147,False,13,1093,0,0.0,2017,6,29,3,0.693147,1.098612,0.693147,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
34725407,122767612,2017-07-21,1,96995,1.386294,False,13,1093,0,0.0,2017,7,21,4,0.693147,1.386294,0.693147,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
34827517,122869722,2017-07-22,1,96995,1.098612,False,13,1093,0,0.0,2017,7,22,5,0.693147,0.693147,0.792168,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False


(35101789, 89)


# III. Train Model

## 1. Create train and test data

In [13]:
# cut-off range
cutoff = data['date'].max() - pd.Timedelta(weeks=12)
# Use time-based hold out, take last 12 weeks data as test data
train = data[data['date'] < cutoff]
test  = data[data['date'] >= cutoff]


## 2. Train XGboost models

In [14]:

stores_list = train['store_nbr'].unique()

models = {}

for s in stores_list:

    df_s = train[train['store_nbr'] == s].copy()

    X = df_s.drop(columns=['unit_sales', 'date', 'id','store_nbr'])
    y = df_s['unit_sales']

    model = XGBRegressor(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.7,
        n_jobs=2
    )

    model.fit(X, y)

    models[s] = model

## 3. Make predictions and evaluate performance

In [15]:
results = []

for s in test['store_nbr'].unique():

    df_s = test[test['store_nbr'] == s].copy()

    X_test = df_s.drop(columns=['unit_sales', 'date', 'id','store_nbr'])
    y_test = df_s['unit_sales']

    model = models.get(s)

    if model is None:
        continue

    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    

    results.append({
        "store": s,
        "rmse": rmse,
        "mse": mse,
        "r2": r2,
        
    })
results_df = pd.DataFrame(results)
display(results_df)
avg_r2 = results_df['r2'].mean()
avg_rmse = results_df['rmse'].mean()
avg_mse = results_df['mse'].mean()

print("Average R2:", avg_r2)
print("Average RMSE:", avg_rmse)
print("Average MSE:", avg_mse)

,store,rmse,mse,r2
0,1,0.486473,0.236656,0.580728
1,2,0.485265,0.235482,0.620778
2,3,0.497969,0.247974,0.742964
3,4,0.485575,0.235783,0.593949
4,5,0.481656,0.231993,0.561741
5,6,0.493817,0.243855,0.632439
6,7,0.494212,0.244246,0.695190
7,8,0.488552,0.238683,0.680383
8,9,0.507826,0.257887,0.644212
9,10,0.479074,0.229512,0.553286


Average R2: 0.610233253598652
Average RMSE: 0.503946276930904
Average MSE: 0.25424868875177287


## 4. Export Dataset for Power BI


### 4.1 Sales by store number

In [16]:
results_full = []

for s in test['store_nbr'].unique():
    df_s = test[test['store_nbr'] == s].copy()
    X_test = df_s.drop(columns=['unit_sales','date','id','store_nbr'])
    # convert back to original sales
    df_s['unit_sales'] = np.expm1(df_s['unit_sales'])
    

    model = models.get(s)
    if model is None:
        continue
    # convert to original sales
    df_s['predicted_sales'] = np.clip(
    np.expm1(model.predict(X_test)), 0, None)
   

    results_full.append(df_s)

final_df = pd.concat(results_full)
agg_df = final_df.groupby(['date','store_nbr']).agg({
    'unit_sales': 'sum',
    'predicted_sales': 'sum'
}).reset_index()


os.makedirs("/kaggle/working/PowerBICSV", exist_ok=True)
data_cleaned_path = "/kaggle/working/PowerBICSV/"
agg_df.to_csv(data_cleaned_path + "powerbi_data.csv", index=False)

### 4.2 Sales by family

In [17]:
results_full = []

for s in test['store_nbr'].unique():
    df_s = test[test['store_nbr'] == s].copy()
    X_test = df_s.drop(columns=['unit_sales','date','id','store_nbr'])
    # convert back to original sales
    df_s['unit_sales'] = np.expm1(df_s['unit_sales'])

    model = models.get(s)
    if model is None:
        continue

    # convert to original sales
    df_s['predicted_sales'] = np.clip(
    np.expm1(model.predict(X_test)), 0, None)

    results_full.append(df_s)

final_df = pd.concat(results_full)
family_cols = [c for c in final_df.columns if c.startswith('family_')]
final_df['family'] = final_df[family_cols].idxmax(axis=1).str.replace('family_', '')
agg_df = final_df.groupby(['date', 'store_nbr', 'family']).agg({
    'unit_sales': 'sum',
    'predicted_sales': 'sum'
}).reset_index()



data_cleaned_path = "/kaggle/working/PowerBICSV/"
agg_df.to_csv(data_cleaned_path + "powerbi_data1.csv", index=False)

In [18]:

# Zip data to download
!zip -r powerBI.zip /kaggle/working/PowerBICSV
from IPython.display import FileLink
FileLink(r'powerBI.zip')


updating: kaggle/working/PowerBICSV/ (stored 0%)
updating: kaggle/working/PowerBICSV/powerbi_data.csv (deflated 63%)
updating: kaggle/working/PowerBICSV/powerbi_data1.csv (deflated 75%)


/kaggle/working/powerBI.zip

## 5. Power BI Visualization for predicted sales and actual sales by store and family product 